# Issue #249 — migration-safe data snapshot

This notebook profiles the same external-drive snapshot and isolated ext4 restore receipt used for the WSL dry run.

In [1]:
import json
from pathlib import Path
import pandas as pd

root = Path.cwd()
evidence = root / 'evidence' / 'bj249_data_portability_v1'
receipt = json.loads((evidence / 'wsl_dry_run_receipt.json').read_text())
manifest = json.loads((evidence / 'MANIFEST.json').read_text())
print('SNAPSHOT_STATUS:', receipt['snapshot_verification']['status'])
print('RESTORE_MATCH:', receipt['restore_verification']['status'])
print('SNAPSHOT_ID:', receipt['snapshot_id'])

SNAPSHOT_STATUS: PASS
RESTORE_MATCH: PASS
SNAPSHOT_ID: bj249-core-20260820-v1


In [2]:
assets = pd.DataFrame([
    {
        'logical_path': asset['logical_path'],
        'kind': asset['kind'],
        'bytes': asset['bytes'],
        'table_count': asset.get('sqlite', {}).get('table_count'),
        'row_count': sum(asset.get('sqlite', {}).get('row_counts', {}).values()),
        'time_range_count': len(asset.get('sqlite', {}).get('time_ranges', [])),
    }
    for asset in manifest['assets']
])
summary = pd.DataFrame([{
    'snapshot_id': receipt['snapshot_id'],
    'asset_count': len(assets),
    'sqlite_count': int((assets.kind == 'sqlite').sum()),
    'raw_or_parquet_count': int((assets.kind == 'file').sum()),
    'total_gib': assets.bytes.sum() / 1024**3,
    'restored_assets': receipt['restore_verification']['restored_asset_count'],
}])
display(summary)
display(assets.sort_values('bytes', ascending=False).head(20))

,snapshot_id,asset_count,sqlite_count,raw_or_parquet_count,total_gib,restored_assets
0,bj249-core-20260820-v1,464,8,456,33.098159,464


,logical_path,kind,bytes,table_count,row_count,time_range_count
0,data/derived/500_bjdc/500_bjdc.sqlite,sqlite,20818624512,26.0,26949131,2
1,data/derived/beidan_xlsm/beidan_xlsm.sqlite,sqlite,6872944640,12.0,64903802,1
3,data/derived/bj268/eu_proxy_backfill_v3/candid...,sqlite,5445255168,11.0,29872263,1
2,data/derived/beidan_xlsm/beidan_modeling_plus_...,sqlite,1087799296,7.0,5812838,1
13,data/raw/beidan.xlsm,file,808551905,NaN,0,0
4,data/beidan_live.db,sqlite,284028928,3.0,1632878,2
73,data/derived/bj238_divergence_materializer_v4/...,file,66918671,NaN,0,0
5,data/beidan_features.db,sqlite,47226880,1.0,190746,1
14,data/raw/beidan_full_3sp.csv,file,19952010,NaN,0,0
6,data/beidan_history.db,sqlite,6758400,2.0,38613,1


In [3]:
time_ranges = pd.DataFrame([
    {'logical_path': asset['logical_path'], **time_range}
    for asset in manifest['assets']
    for time_range in asset.get('sqlite', {}).get('time_ranges', [])
])
display(time_ranges)
assert receipt['snapshot_verification']['manifest_sha256'] == receipt['manifest_sha256']
assert receipt['restore_verification']['restored_asset_count'] == len(assets)
assert receipt['source_mutation_check'].startswith('PASS')

,logical_path,column,maximum,minimum,non_null_rows,table
0,data/derived/500_bjdc/500_bjdc.sqlite,schedule_date,2026-08-20,2013-12-28,76123,matches
1,data/derived/500_bjdc/500_bjdc.sqlite,fetched_at_utc,2026-08-20T03:53:05+00:00,2026-07-16T13:05:41+00:00,24733176,live_market_sp_snapshots
2,data/derived/beidan_xlsm/beidan_xlsm.sqlite,kickoff_ts,2025-07-13 03:00:00,1899-12-30 08:00:00,192709,matches
3,data/derived/beidan_xlsm/beidan_modeling_plus_...,kickoff_ts,2026-07-27 03:15:00,1899-12-30 08:00:00,198710,matches
4,data/derived/bj268/eu_proxy_backfill_v3/candid...,kickoff_ts,2026-07-27 03:15:00,1899-12-30 08:00:00,198710,matches
5,data/beidan_live.db,kickoff_ts,2026-08-21T08:30:00,2026-04-30T19:30:00,5494,matches
6,data/beidan_live.db,snapshot_ts,2026-08-20T04:09:25Z,2026-05-02T00:57:41Z,1623858,snapshots
7,data/beidan_features.db,kickoff,2025-04-15T08:30:00,08:00:00,190746,matches
8,data/beidan_history.db,kickoff_ts,2026-12-31 16:00:00,2024-01-01 14:00:00,38176,historical_matches
